In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

train_data = pd.read_csv('./data/train.csv') # Importing training data

X = train_data.drop(["time", "Y1", "Y2"], axis = 1) # Losing the time and target columns

new_train_data = pd.read_csv("./data/train_new.csv")

X = pd.concat((X, new_train_data), axis = 1, join = "inner")

# Setting target variables
y1 = train_data["Y1"]

y2 = train_data["Y2"]

X, X_test, y1, y1_test, y2, y2_test = train_test_split(X, y1, y2)

In [2]:
from sklearn.decomposition import PCA

# Applying PCA to training set
X[["A", "B", "C", "D", "E", "F", "G", "H", "I", "J", "K", "L", "M", "N"]] = (X.drop(["O", "P"], axis = 1) - X.drop(["O", "P"], axis = 1).mean(axis=0)) / X.drop(["O", "P"], axis = 1).std(axis=0)

pca = PCA(n_components= 3)

X_pca = pca.fit_transform(X.drop(["O", "P"], axis = 1))

X["PC1"] = X_pca[:,0]
X["PC2"] = X_pca[:,1]
X["PC3"] = X_pca[:,2]

In [3]:
loadings = pd.DataFrame(
    pca.components_.T,
    columns = ["PC1", "PC2", "PC3"],
    index = X.drop(["PC1", "PC2", "PC3", "O", "P"], axis = 1).columns
)

In [4]:
X_test[["A", "B", "C", "D", "E", "F", "G", "H", "I", "J", "K", "L", "M", "N"]] = (X_test.drop(["O", "P"], axis = 1) - X_test.drop(["O", "P"], axis = 1).mean(axis=0)) / X_test.drop(["O", "P"], axis = 1).std(axis=0)

X_pca = X_test.drop(["O", "P"], axis = 1).dot(loadings)

X_test["PC1"] = X_pca["PC1"]
X_test["PC2"] = X_pca["PC2"]
X_test["PC3"] = X_pca["PC3"]

In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression

featuresX1 = ["PC1", "G", "H"]
featuresX2 = ["A", "B", "D", "F", "G", "K", "PC1", "PC2"]

# Choosing Features
X1_train = X[featuresX1]

# Modelling
modelY1 = RandomForestRegressor(n_estimators=300, n_jobs = -1)

modelY1.fit(X1_train, y1)

# Choosing Features

X2_train = X[featuresX2]

# Modelling
modelY2 = RandomForestRegressor(n_estimators=300, n_jobs = -1)

modelY2.fit(X2_train, y2)

,n_estimators,300
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [6]:
y1_train_pred = modelY1.predict(X_test[featuresX1])
y2_train_pred = modelY2.predict(X_test[featuresX2])

print(f"Predicted score : {(r2_score(y2_train_pred, y2_test) + r2_score(y1_train_pred, y1_test))/2}")
print(f"Score Y1 : {r2_score(y1_train_pred, y1_test)} \nScore Y2 : {r2_score(y2_train_pred, y2_test)}")

Predicted score : 0.6421124328649378
Score Y1 : 0.670141239677483 
Score Y2 : 0.6140836260523926


In [7]:
from sklearn.decomposition import PCA

test_data = pd.read_csv('./data/test.csv') # Importing training data

X = test_data.drop(["time", "id"], axis = 1) # Losing the time column

new_train_data = pd.read_csv("./data/train_new.csv")

X = pd.concat((X, new_train_data), axis = 1, join = "inner")

# Applying PCA to training set
X[["A", "B", "C", "D", "E", "F", "G", "H", "I", "J", "K", "L", "M", "N"]] = (X.drop(["O", "P"], axis = 1) - X.drop(["O", "P"], axis = 1).mean(axis=0)) / X.drop(["O", "P"], axis = 1).std(axis=0)

X_pca = X.drop(["O", "P"], axis = 1).dot(loadings)

X["PC1"] = X_pca["PC1"]
X["PC2"] = X_pca["PC2"]
X["PC3"] = X_pca["PC3"]

X1 = X[featuresX1]

y1_pred = pd.DataFrame(modelY1.predict(X1), index = test_data.id, columns = ["Y1"])

X2 = X[featuresX2]

y2_pred = pd.DataFrame(modelY2.predict(X2), index = test_data.id, columns = ["Y2"])

out = pd.concat((y1_pred, y2_pred), axis = 1)

out.to_csv("./data/predictions.csv") # Scores 0.5888


#[["G", "M", "J", "C", "E", "H", "N", "PC1"]]
#[["A" ,"PC2", "K", "B", "D", "F", "I", "K", "L"]]

# X_pca = X.dot(loadings)

# X = pd.concat((X, X_pca), axis = 1, join = "inner")

In [17]:
best_pred = pd.read_csv("./data/predictions_0-69.csv").drop("id", axis = 1)
second_best_pred = pd.read_csv("./data/pred_0-685.csv").drop("id", axis = 1)
out = pd.read_csv("./data/predictions-hypo-4.csv").drop("id", axis = 1)
big_gaussian = pd.read_csv("./data/predictions_Big_Gaussian.csv").drop("id", axis = 1)

print(r2_score(out, best_pred))
print(r2_score(out, second_best_pred))
print(r2_score(out, big_gaussian))

print(r2_score(second_best_pred, best_pred))
print(r2_score(big_gaussian, best_pred))
print(r2_score(big_gaussian, second_best_pred))

# Hypo 1 - Feature selection:
#[["G", "M", "J", "C", "E", "H", "N", "PC1"]]
#[["A" ,"PC2", "K", "B", "D", "F", "I", "K", "L"]]

# Hypo 2 - no feature selection

# Hypo 3 - models.ipynb 

# Hypo 4 - Feature Y1, no Feature Y2

0.9734244854728887
0.9862424348633121
0.9833678695430099
0.9765123572310014
0.9764528097248427
0.9719567583801074
